# 18B — V5 Deterministic Identity Repair

This notebook repairs roster membership after the identity-linkage audit found same-name collisions.

**Conservative frozen rule:** only `PASS_EXACT_DOB` selected subjects are retained.

Therefore these selected rows are replaced:

- `PASS_LEGACY_WAVE_A`
- `LIKELY_IDENTITY_MISMATCH`
- every `REVIEW_*`

Replacement candidates must already be `PASS_EXACT_DOB` in the audited candidate pool and must stay in the same axis.

The notebook does **not** load Batch 1-3 event results.  
It preserves each DEV `subject_id`, axis, batch and research order, so only the identity occupying a bad slot changes.

CONFIRM is repaired locally but remains sealed.


In [1]:

from pathlib import Path
from datetime import datetime
import hashlib, json, re, unicodedata
import numpy as np
import pandas as pd
from IPython.display import display

NOTEBOOK_VERSION="SAJU_ML_V5_DETERMINISTIC_IDENTITY_REPAIR_20260817"
SELECTION_SEED=2026081702
KEEP_STATUS="PASS_EXACT_DOB"

def find_repo_root(start=None):
    p=Path(start or Path.cwd()).resolve()
    for c in [p]+list(p.parents):
        if (c/"saju_engine.py").exists():
            return c
    raise FileNotFoundError("Run inside Chartpalja repo.")

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""):
            h.update(chunk)
    return h.hexdigest()

def norm_name(x):
    s=unicodedata.normalize("NFKD",str(x))
    s="".join(c for c in s if not unicodedata.combining(c))
    s=s.casefold()
    return re.sub(r"[^a-z0-9]+","",s)

def dkey(axis, qid, name):
    raw=f"{SELECTION_SEED}|{axis}|{qid}|{norm_name(name)}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()

ROOT=find_repo_root()

FINAL=ROOT/"research/ml/artifacts/v5_final_roster"
AUDIT=ROOT/"research/ml/artifacts/v5_identity_linkage_audit"
EVENT=ROOT/"research/ml/artifacts/v5_event_collection"
CORPUS=ROOT/"research/ml_corpus/v5_ground_truth"

OUT=ROOT/"research/ml/artifacts/v5_identity_repair"
EVENT_OUT=ROOT/"research/ml/artifacts/v5_event_collection_repaired"
BATCH_OUT=EVENT_OUT/"batches"

OUT.mkdir(parents=True,exist_ok=True)
EVENT_OUT.mkdir(parents=True,exist_ok=True)
BATCH_OUT.mkdir(parents=True,exist_ok=True)

DEV_PATH=FINAL/"V5_DEV_SUBJECT_ROSTER_160.csv"
CONFIRM_PATH=FINAL/"V5_CONFIRM_SUBJECT_ROSTER_80_SEALED.csv"
FREEZE_PATH=FINAL/"V5_FINAL_ROSTER_FREEZE_DECISION.json"

DEV_AUDIT_PATH=AUDIT/"V5_IDENTITY_LINKAGE_AUDIT_DEV.csv"
CONFIRM_AUDIT_PATH=AUDIT/"V5_IDENTITY_LINKAGE_AUDIT_CONFIRM_SEALED.csv"
POOL_AUDIT_PATH=AUDIT/"V5_IDENTITY_LINKAGE_AUDIT_CANDIDATE_POOL.csv"
AUDIT_DECISION_PATH=AUDIT/"V5_IDENTITY_LINKAGE_AUDIT_DECISION.json"

WORKLIST_PATH=EVENT/"V5_DEV_EVENT_RESEARCH_WORKLIST_8BATCH.csv"
PROTOCOL_PATH=CORPUS/"V5_DETERMINISTIC_IDENTITY_REPAIR_PROTOCOL.json"

for p in [
    DEV_PATH,CONFIRM_PATH,FREEZE_PATH,
    DEV_AUDIT_PATH,CONFIRM_AUDIT_PATH,POOL_AUDIT_PATH,AUDIT_DECISION_PATH,
    WORKLIST_PATH,PROTOCOL_PATH
]:
    if not p.exists():
        raise FileNotFoundError(p)

dev=pd.read_csv(DEV_PATH)
confirm=pd.read_csv(CONFIRM_PATH)
adev=pd.read_csv(DEV_AUDIT_PATH)
aconfirm=pd.read_csv(CONFIRM_AUDIT_PATH)
apool=pd.read_csv(POOL_AUDIT_PATH)
worklist=pd.read_csv(WORKLIST_PATH)

freeze=json.load(open(FREEZE_PATH,encoding="utf-8"))
audit_decision=json.load(open(AUDIT_DECISION_PATH,encoding="utf-8"))
protocol=json.load(open(PROTOCOL_PATH,encoding="utf-8"))

assert freeze["status"]=="V5_FINAL_DEV_AND_CONFIRM_ROSTERS_FROZEN_READY_FOR_DEV_EVENT_COLLECTION"
assert audit_decision["status"]=="V5_IDENTITY_LINKAGE_AUDIT_REPAIR_REQUIRED"
assert sha256_file(DEV_PATH)==freeze["dev_roster_sha256"]
assert sha256_file(CONFIRM_PATH)==freeze["confirm_roster_sha256"]
assert sha256_file(DEV_AUDIT_PATH)==audit_decision["DEV_audit_sha256"]
assert sha256_file(CONFIRM_AUDIT_PATH)==audit_decision["CONFIRM_sealed_audit_sha256"]
assert sha256_file(POOL_AUDIT_PATH)==audit_decision["candidate_pool_audit_sha256"]
assert protocol["status"]=="PREDECLARED_BEFORE_MEMBERSHIP_REPAIR"

assert len(dev)==160 and len(confirm)==80
assert len(worklist)==160
assert set(worklist.subject_id)==set(dev.subject_id)

# Deliberately no paths to Batch 1-3 event outputs are defined here.
print("IDENTITY REPAIR PREFLIGHT PASS")
print("No event-result path loaded.")


IDENTITY REPAIR PREFLIGHT PASS
No event-result path loaded.


## 1. Determine slots that must be replaced

In [2]:

dev_status=adev.set_index("subject_id")["linkage_status"]
confirm_status=aconfirm.set_index("subject_id")["linkage_status"]

assert set(dev.subject_id)==set(dev_status.index)
assert set(confirm.subject_id)==set(confirm_status.index)

dev_bad=dev[dev.subject_id.map(dev_status)!=KEEP_STATUS].copy()
confirm_bad=confirm[confirm.subject_id.map(confirm_status)!=KEEP_STATUS].copy()

dev_bad["old_linkage_status"]=dev_bad.subject_id.map(dev_status)
confirm_bad["old_linkage_status"]=confirm_bad.subject_id.map(confirm_status)

print("DEV replacements required:",len(dev_bad))
display(
    dev_bad.groupby(["preassigned_axis","old_linkage_status"])
    .size().rename("n").reset_index()
)

print("CONFIRM replacements required (aggregate only):",len(confirm_bad))
display(
    confirm_bad.groupby(["preassigned_axis","old_linkage_status"])
    .size().rename("n").reset_index()
)

# Current selected identities are blocked from replacement pool.
selected_norm=set(dev.name.map(norm_name)) | set(confirm.name.map(norm_name))
selected_qids=set(
    str(x) for x in pd.concat([dev.wikidata_id,confirm.wikidata_id]).dropna()
    if str(x).startswith("Q")
)


DEV replacements required: 27


,preassigned_axis,old_linkage_status,n
0,COMPETITIVE,LIKELY_IDENTITY_MISMATCH,2
1,COMPETITIVE,PASS_LEGACY_WAVE_A,7
2,COMPETITIVE,REVIEW_DOB_DISCREPANCY_SAME_PLACE,1
3,COMPETITIVE,REVIEW_DOB_DISCREPANCY_UNKNOWN_PLACE,1
4,COMPETITIVE,REVIEW_MISSING_OR_LOW_PRECISION_DOB,1
5,PROJECT,PASS_LEGACY_WAVE_A,2
6,PROJECT,REVIEW_DOB_DISCREPANCY_SAME_PLACE,1
7,STATUS,LIKELY_IDENTITY_MISMATCH,4
8,STATUS,PASS_LEGACY_WAVE_A,6
9,STATUS,REVIEW_DOB_DISCREPANCY_SAME_PLACE,1


CONFIRM replacements required (aggregate only): 16


,preassigned_axis,old_linkage_status,n
0,COMPETITIVE,LIKELY_IDENTITY_MISMATCH,1
1,COMPETITIVE,PASS_LEGACY_WAVE_A,3
2,COMPETITIVE,REVIEW_DOB_DISCREPANCY_SAME_PLACE,2
3,PROJECT,LIKELY_IDENTITY_MISMATCH,1
4,PROJECT,PASS_LEGACY_WAVE_A,2
5,STATUS,LIKELY_IDENTITY_MISMATCH,1
6,STATUS,PASS_LEGACY_WAVE_A,2
7,STATUS,REVIEW_DOB_DISCREPANCY_SAME_PLACE,2
8,STATUS,REVIEW_MISSING_OR_LOW_PRECISION_DOB,2


## 2. Build exact-DOB-only deterministic replacement pool

In [3]:

cand=apool[
    (apool.linkage_status==KEEP_STATUS)
    & apool.wikidata_id.notna()
].copy()

cand["norm_name_repair"]=cand.name.map(norm_name)
cand=cand[
    ~cand.norm_name_repair.isin(selected_norm)
    & ~cand.wikidata_id.astype(str).isin(selected_qids)
].copy()

cand["repair_key"]=cand.apply(
    lambda r:dkey(r.axis,str(r.wikidata_id),r["name"]),
    axis=1
)

# Deduplicate identity conservatively.
cand=(
    cand.sort_values(["axis","repair_key","wikidata_id","norm_name_repair"])
    .drop_duplicates(["wikidata_id"])
    .drop_duplicates(["axis","norm_name_repair"])
    .reset_index(drop=True)
)

need_dev=dev_bad.preassigned_axis.value_counts().to_dict()
need_confirm=confirm_bad.preassigned_axis.value_counts().to_dict()
need_total={
    axis:int(need_dev.get(axis,0))+int(need_confirm.get(axis,0))
    for axis in ["COMPETITIVE","PROJECT","STATUS"]
}
avail=cand.axis.value_counts().to_dict()

suff=pd.DataFrame([
    {
        "axis":axis,
        "DEV_replacements":int(need_dev.get(axis,0)),
        "CONFIRM_replacements":int(need_confirm.get(axis,0)),
        "total_needed":int(need_total[axis]),
        "exact_DOB_candidates_available":int(avail.get(axis,0)),
        "sufficient":int(avail.get(axis,0))>=int(need_total[axis])
    }
    for axis in ["COMPETITIVE","PROJECT","STATUS"]
])
display(suff)
assert suff.sufficient.all(), suff.to_dict(orient="records")

print("Exact-DOB replacement-pool sufficiency PASS")


,axis,DEV_replacements,CONFIRM_replacements,total_needed,exact_DOB_candidates_available,sufficient
0,COMPETITIVE,12,6,18,113,True
1,PROJECT,3,3,6,851,True
2,STATUS,12,7,19,55,True


Exact-DOB replacement-pool sufficiency PASS


## 3. Deterministically assign replacements to fixed DEV/CONFIRM slots

In [4]:

def allocate_axis(axis, dev_slots, confirm_slots):
    pool=cand[cand.axis==axis].sort_values(
        ["repair_key","wikidata_id","norm_name_repair"]
    ).reset_index(drop=True)

    n_dev=len(dev_slots)
    n_confirm=len(confirm_slots)
    chosen=pool.head(n_dev+n_confirm).copy()

    if len(chosen)!=(n_dev+n_confirm):
        raise RuntimeError(f"{axis}: insufficient candidates")

    # Slot mapping is deterministic and does not depend on events.
    dslots=dev_slots.sort_values("subject_id").reset_index(drop=True)
    cslots=confirm_slots.sort_values("subject_id").reset_index(drop=True)

    dchosen=chosen.iloc[:n_dev].reset_index(drop=True)
    cchosen=chosen.iloc[n_dev:n_dev+n_confirm].reset_index(drop=True)

    return dslots,dchosen,cslots,cchosen

dev_maps=[]
confirm_maps=[]

for axis in ["COMPETITIVE","PROJECT","STATUS"]:
    ds=dev_bad[dev_bad.preassigned_axis==axis].copy()
    cs=confirm_bad[confirm_bad.preassigned_axis==axis].copy()

    dslots,dchosen,cslots,cchosen=allocate_axis(axis,ds,cs)

    for i in range(len(dslots)):
        old=dslots.iloc[i]
        new=dchosen.iloc[i]
        dev_maps.append({
            "subject_id":old.subject_id,
            "split":"DEV",
            "axis":axis,
            "old_name":old["name"],
            "old_birth_date":old.birth_date,
            "old_birth_place":old.birth_place,
            "old_wikidata_id":old.get("wikidata_id",np.nan),
            "old_linkage_status":old.old_linkage_status,
            "new_name":new["name"],
            "new_birth_date":new.birth_date,
            "new_birth_place":new.birth_place,
            "new_wikidata_id":new.wikidata_id,
            "new_linkage_status":new.linkage_status,
            "replacement_key":new.repair_key,
        })

    for i in range(len(cslots)):
        old=cslots.iloc[i]
        new=cchosen.iloc[i]
        confirm_maps.append({
            "subject_id":old.subject_id,
            "split":"CONFIRM",
            "axis":axis,
            "old_name":old["name"],
            "old_birth_date":old.birth_date,
            "old_birth_place":old.birth_place,
            "old_wikidata_id":old.get("wikidata_id",np.nan),
            "old_linkage_status":old.old_linkage_status,
            "new_name":new["name"],
            "new_birth_date":new.birth_date,
            "new_birth_place":new.birth_place,
            "new_wikidata_id":new.wikidata_id,
            "new_linkage_status":new.linkage_status,
            "replacement_key":new.repair_key,
        })

dev_map=pd.DataFrame(dev_maps)
confirm_map=pd.DataFrame(confirm_maps)

assert (dev_map.new_linkage_status==KEEP_STATUS).all()
assert (confirm_map.new_linkage_status==KEEP_STATUS).all()
assert set(dev_map.new_wikidata_id).isdisjoint(set(confirm_map.new_wikidata_id))

print("DEV replacement map")
display(dev_map.groupby(["axis","old_linkage_status"]).size().rename("n").reset_index())
print("CONFIRM replacement map aggregate")
display(confirm_map.groupby(["axis","old_linkage_status"]).size().rename("n").reset_index())


DEV replacement map


,axis,old_linkage_status,n
0,COMPETITIVE,LIKELY_IDENTITY_MISMATCH,2
1,COMPETITIVE,PASS_LEGACY_WAVE_A,7
2,COMPETITIVE,REVIEW_DOB_DISCREPANCY_SAME_PLACE,1
3,COMPETITIVE,REVIEW_DOB_DISCREPANCY_UNKNOWN_PLACE,1
4,COMPETITIVE,REVIEW_MISSING_OR_LOW_PRECISION_DOB,1
5,PROJECT,PASS_LEGACY_WAVE_A,2
6,PROJECT,REVIEW_DOB_DISCREPANCY_SAME_PLACE,1
7,STATUS,LIKELY_IDENTITY_MISMATCH,4
8,STATUS,PASS_LEGACY_WAVE_A,6
9,STATUS,REVIEW_DOB_DISCREPANCY_SAME_PLACE,1


CONFIRM replacement map aggregate


,axis,old_linkage_status,n
0,COMPETITIVE,LIKELY_IDENTITY_MISMATCH,1
1,COMPETITIVE,PASS_LEGACY_WAVE_A,3
2,COMPETITIVE,REVIEW_DOB_DISCREPANCY_SAME_PLACE,2
3,PROJECT,LIKELY_IDENTITY_MISMATCH,1
4,PROJECT,PASS_LEGACY_WAVE_A,2
5,STATUS,LIKELY_IDENTITY_MISMATCH,1
6,STATUS,PASS_LEGACY_WAVE_A,2
7,STATUS,REVIEW_DOB_DISCREPANCY_SAME_PLACE,2
8,STATUS,REVIEW_MISSING_OR_LOW_PRECISION_DOB,2


## 4. Apply replacement identities while preserving subject slots

In [5]:

# Candidate columns allowed to replace identity/birth metadata.
identity_cols=[
    "name","source_name","gender",
    "birth_date","birth_time","utc_offset","birth_place","latitude","longitude",
    "rodden_rating","source_row_key","birth_source",
    "candidate_source","role_family","role_qid","wikidata_id","wikidata_sitelinks",
    "seed_origin","selection_information_used"
]

cand_by_qid=cand.set_index(cand.wikidata_id.astype(str),drop=False)

def repaired_roster(old_roster, rep_map):
    x=old_roster.copy()
    for _,m in rep_map.iterrows():
        qid=str(m.new_wikidata_id)
        new=cand_by_qid.loc[qid]
        mask=x.subject_id==m.subject_id
        assert mask.sum()==1

        old_axis=str(x.loc[mask,"preassigned_axis"].iloc[0])
        assert old_axis==str(new.axis)

        for c in identity_cols:
            if c in x.columns and c in new.index:
                x.loc[mask,c]=new[c]

        # Slot-level fields are immutable.
        x.loc[mask,"preassigned_axis"]=old_axis
        x.loc[mask,"split"]=str(x.loc[mask,"split"].iloc[0])
        x.loc[mask,"event_collection_started"]=False
        x.loc[mask,"astrology_scored"]=False

    return x

rdev=repaired_roster(dev,dev_map)
rconfirm=repaired_roster(confirm,confirm_map)

assert len(rdev)==160 and rdev.subject_id.nunique()==160
assert len(rconfirm)==80 and rconfirm.subject_id.nunique()==80
assert rdev.preassigned_axis.value_counts().to_dict()==dev.preassigned_axis.value_counts().to_dict()
assert rconfirm.preassigned_axis.value_counts().to_dict()==confirm.preassigned_axis.value_counts().to_dict()
assert set(rdev.name.map(norm_name)).isdisjoint(set(rconfirm.name.map(norm_name)))
assert set(rdev.wikidata_id.astype(str)).isdisjoint(set(rconfirm.wikidata_id.astype(str)))

# Every repaired selected row must have exact-DOB audit evidence:
old_dev_status=adev.set_index("subject_id")["linkage_status"].to_dict()
old_confirm_status=aconfirm.set_index("subject_id")["linkage_status"].to_dict()
replaced_dev=set(dev_map.subject_id)
replaced_confirm=set(confirm_map.subject_id)

for sid in rdev.subject_id:
    if sid in replaced_dev:
        assert str(rdev.loc[rdev.subject_id==sid,"wikidata_id"].iloc[0]) in set(cand[cand.linkage_status==KEEP_STATUS].wikidata_id.astype(str))
    else:
        assert old_dev_status[sid]==KEEP_STATUS

for sid in rconfirm.subject_id:
    if sid in replaced_confirm:
        assert str(rconfirm.loc[rconfirm.subject_id==sid,"wikidata_id"].iloc[0]) in set(cand[cand.linkage_status==KEEP_STATUS].wikidata_id.astype(str))
    else:
        assert old_confirm_status[sid]==KEEP_STATUS

print("Repaired roster identity rule PASS: every selected slot is exact-DOB validated.")


Repaired roster identity rule PASS: every selected slot is exact-DOB validated.


## 5. Repair DEV worklist and batch subject files without loading event results

In [6]:

# Preserve scheduling columns from old worklist, refresh identity columns from repaired DEV roster.
schedule_cols=[
    "batch_id","research_order_in_batch","subject_id","batch_hash"
]
schedule=worklist[schedule_cols].copy()

identity_out_cols=[
    "subject_id","name","preassigned_axis","candidate_source",
    "birth_date","birth_place","gender"
]
rw=schedule.merge(
    rdev[identity_out_cols],
    on="subject_id",
    how="left",
    validate="one_to_one"
)

assert len(rw)==160
assert rw.name.notna().all()
assert rw.batch_id.nunique()==8

repaired_worklist_path=EVENT_OUT/"V5_DEV_EVENT_RESEARCH_WORKLIST_8BATCH_REPAIRED.csv"
rw.to_csv(repaired_worklist_path,index=False)

for b in range(1,9):
    wb=rw[rw.batch_id==b].sort_values("research_order_in_batch").copy()
    wb.to_csv(
        BATCH_OUT/f"V5_DEV_EVENT_BATCH_{b:02d}_SUBJECTS_REPAIRED.csv",
        index=False
    )

slot_info=worklist[["subject_id","batch_id","research_order_in_batch"]].copy()
invalidated=dev_map.merge(slot_info,on="subject_id",how="left",validate="one_to_one")
invalidated["existing_event_rows_action"]="DELETE_ALL_OLD_EVENT_ROWS_FOR_SUBJECT_ID_AND_RESEARCH_NEW_IDENTITY"
invalidated["reason"]="IDENTITY_SLOT_REPLACED"
invalidated_path=EVENT_OUT/"V5_EVENT_RESEARCH_INVALIDATED_SLOTS.csv"
invalidated.to_csv(invalidated_path,index=False)

print("Repaired batch counts:")
display(
    rw.groupby(["batch_id","preassigned_axis"]).size().unstack(fill_value=0)
)
print("Previously researched Batch 1-3 slots invalidated:",
      int(invalidated.batch_id.isin([1,2,3]).sum()))


Repaired batch counts:


preassigned_axis,COMPETITIVE,PROJECT,STATUS
batch_id,,,
1,5,7,9
2,5,7,9
3,5,6,9
4,5,6,9
5,5,6,9
6,5,6,9
7,5,6,8
8,5,6,8


Previously researched Batch 1-3 slots invalidated: 6


## 6. Freeze repair outputs

In [7]:

dev_path=OUT/"V5_DEV_SUBJECT_ROSTER_160_REPAIRED.csv"
confirm_path=OUT/"V5_CONFIRM_SUBJECT_ROSTER_80_REPAIRED_SEALED.csv"
dev_map_path=OUT/"V5_IDENTITY_REPAIR_MAP_DEV.csv"
confirm_map_path=OUT/"V5_IDENTITY_REPAIR_MAP_CONFIRM_SEALED.csv"

rdev.to_csv(dev_path,index=False)
rconfirm.to_csv(confirm_path,index=False)
dev_map.to_csv(dev_map_path,index=False)
confirm_map.to_csv(confirm_map_path,index=False)

decision={
    "version":"V5_IDENTITY_REPAIR_DECISION_V1",
    "notebook_version":NOTEBOOK_VERSION,
    "created_at":datetime.now().isoformat(timespec="seconds"),
    "status":"V5_IDENTITY_REPAIR_APPLIED_READY_FOR_POST_REPAIR_VALIDATION",
    "retention_rule":"PASS_EXACT_DOB only",
    "DEV_replaced_n":int(len(dev_map)),
    "CONFIRM_replaced_n":int(len(confirm_map)),
    "DEV_replacements_by_axis":dev_map.axis.value_counts().to_dict(),
    "CONFIRM_replacements_by_axis":confirm_map.axis.value_counts().to_dict(),
    "old_DEV_sha256":sha256_file(DEV_PATH),
    "old_CONFIRM_sha256":sha256_file(CONFIRM_PATH),
    "repaired_DEV_sha256":sha256_file(dev_path),
    "repaired_CONFIRM_sha256":sha256_file(confirm_path),
    "DEV_repair_map_sha256":sha256_file(dev_map_path),
    "CONFIRM_repair_map_sha256":sha256_file(confirm_map_path),
    "repaired_worklist_sha256":sha256_file(repaired_worklist_path),
    "invalidated_slots_sha256":sha256_file(invalidated_path),
    "audit_decision_sha256":sha256_file(AUDIT_DECISION_PATH),
    "protocol_sha256":sha256_file(PROTOCOL_PATH),
    "selection_seed":SELECTION_SEED,
    "rules":{
        "event_results_loaded":False,
        "chronology_loaded":False,
        "pairability_loaded":False,
        "astrology_loaded":False,
        "control_loaded":False,
        "subject_id_preserved":True,
        "axis_preserved":True,
        "batch_schedule_preserved":True,
        "confirm_details_remain_sealed":True
    },
    "event_research_may_resume":False,
    "batch_04_may_resume":False,
    "next_rule":(
        "Run post-repair validation and sanitize Batch 1-3 event outputs using "
        "V5_EVENT_RESEARCH_INVALIDATED_SLOTS.csv. Do not resume Batch 4 yet."
    )
}
decision_path=OUT/"V5_IDENTITY_REPAIR_DECISION.json"
json.dump(decision,open(decision_path,"w",encoding="utf-8"),ensure_ascii=False,indent=2)

print(json.dumps(decision,ensure_ascii=False,indent=2))
print()
print("DO NOT send repaired CONFIRM roster or CONFIRM repair map.")


{
  "version": "V5_IDENTITY_REPAIR_DECISION_V1",
  "notebook_version": "SAJU_ML_V5_DETERMINISTIC_IDENTITY_REPAIR_20260817",
  "created_at": "2026-08-17T03:36:33",
  "status": "V5_IDENTITY_REPAIR_APPLIED_READY_FOR_POST_REPAIR_VALIDATION",
  "retention_rule": "PASS_EXACT_DOB only",
  "DEV_replaced_n": 27,
  "CONFIRM_replaced_n": 16,
  "DEV_replacements_by_axis": {
    "COMPETITIVE": 12,
    "STATUS": 12,
    "PROJECT": 3
  },
  "CONFIRM_replacements_by_axis": {
    "STATUS": 7,
    "COMPETITIVE": 6,
    "PROJECT": 3
  },
  "old_DEV_sha256": "a7c5a8cbb9855f952d7fbbde7ed6c358d0053dfbec8b2b09f66430413f8a1316",
  "old_CONFIRM_sha256": "3144f52a8e9304c5cf1773a1dc346b58a1e41fa6af784b3a4ea64a6d318dbfac",
  "repaired_DEV_sha256": "f40e8fc2021be911ef2a15012c5eee3128901d820d08fd786c51b15e9f903bd5",
  "repaired_CONFIRM_sha256": "9c22b01e9af49ce40114a7043ba5ae3a3e1179e42a8c41134f8ef5e979462e1a",
  "DEV_repair_map_sha256": "71c9092f42990c1b3d16b3dd7cc6894a90e6582e70e70a28326f3f924af55c08",
  "CONFIRM

## Send back after Run All

Send exactly:

```text
V5_IDENTITY_REPAIR_DECISION.json
V5_DEV_SUBJECT_ROSTER_160_REPAIRED.csv
V5_IDENTITY_REPAIR_MAP_DEV.csv
V5_EVENT_RESEARCH_INVALIDATED_SLOTS.csv
```

Keep sealed locally:

```text
V5_CONFIRM_SUBJECT_ROSTER_80_REPAIRED_SEALED.csv
V5_IDENTITY_REPAIR_MAP_CONFIRM_SEALED.csv
```

Do not resume Batch 4 yet. The next notebook will validate the repair and sanitize Batch 1-3 event results.
